# Quantization Before (QBT)

In [3]:
import torch
import torch.nn as nn
from transformers import BitsAndBytesConfig

In [6]:
model = nn.Sequential(
    nn.Linear(1, 100),
    nn.ReLU()
)

# Or

model2 = nn.Sequential(
    nn.Linear(1, 100, dtype=torch.float16),
    nn.ReLU()
)
print(f'Model 1 (float32)')
total_params_size = sum(p.nelement() * p.element_size() for p in model.parameters())
print(f"Total size: {total_params_size} bytes")
print(f"Size in KB: {total_params_size / 1024:.2f} KB")

print(f'=' * 50)
print(f'Model 2 (float16)')
total_params_size = sum(p.nelement() * p.element_size() for p in model2.parameters())
print(f"Total size: {total_params_size} bytes")
print(f"Size in KB: {total_params_size / 1024:.2f} KB")

Model 1 (float32)
Total size: 800 bytes
Size in KB: 0.78 KB
Model 2 (float16)
Total size: 400 bytes
Size in KB: 0.39 KB


# Quantizatiom During (QDT) (AMP)

here the main it's to use the `torch.cuda.amp` for that, a part of torch that are dedicated for quantization in the training

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler


class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


def get_dataloader(n_samples=1024, batch_size=64):
    X = torch.randn(n_samples, 128)
    y = torch.randint(0, 10, (n_samples,))
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)


# ─── Train without mixed precision (baseline) ───────────────────────────────────
def train_fp32(model, dataloader, optimizer, criterion, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X, y in dataloader:
            X, y = X.cuda(), y.cuda()

            optimizer.zero_grad()
            output = model(X)           # float32
            loss = criterion(output, y) # float32
            loss.backward()             # gradientes em float32
            optimizer.step()

            total_loss += loss.item()

        print(f"[FP32] Epoch {epoch+1} | Loss: {total_loss / len(dataloader):.4f}")


# ─── Train with mixed precision (AMP) ────────────────────────────────────────
def train_amp(model, dataloader, optimizer, criterion, epochs=3):
    model.train()

    # GradScaler: evita underflow dos gradientes em float16
    # O scaler multiplica a loss por um fator alto antes do backward,
    # mantendo os gradientes dentro do range representável pelo float16.
    scaler = GradScaler()

    for epoch in range(epochs):
        total_loss = 0
        for X, y in dataloader:
            X, y = X.cuda(), y.cuda()

            optimizer.zero_grad()

            # autocast: converte automaticamente as ops elegíveis para float16.
            # Os pesos master continuam em float32; o autocast só age no forward.
            with autocast(dtype=torch.float16): # ou with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                output = model(X)           # ops em float16
                loss = criterion(output, y) # loss em float16

            # scaler.scale: multiplica a loss pelo fator de escala antes do backward,
            # evitando que gradientes pequenos virem zero (underflow) em float16.
            scaler.scale(loss).backward()

            # scaler.step: reverte o scaling dos gradientes (divide pelo fator),
            # verifica se há inf/nan, e só então chama optimizer.step() em float32.
            scaler.step(optimizer)

            # scaler.update: ajusta o fator de escala dinamicamente para o próximo step.
            # Se detectou inf/nan, reduz o fator; se está estável, pode aumentar.
            scaler.update()

            total_loss += loss.item()

        print(f"[AMP]  Epoch {epoch+1} | Loss: {total_loss / len(dataloader):.4f} | Scale: {scaler.get_scale()}")


# ─── Main ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Usando: {device}\n")

    dataloader = get_dataloader()
    criterion = nn.CrossEntropyLoss()

    # Baseline FP32
    model_fp32 = SimpleModel().to(device)
    optimizer_fp32 = torch.optim.Adam(model_fp32.parameters(), lr=1e-3)
    train_fp32(model_fp32, dataloader, optimizer_fp32, criterion)

    print()

    # Mixed Precision (AMP)
    model_amp = SimpleModel().to(device)
    optimizer_amp = torch.optim.Adam(model_amp.parameters(), lr=1e-3)
    train_amp(model_amp, dataloader, optimizer_amp, criterion)

# Quantization After (QAT)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 1. Config of bitsandbytes 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  #Define the new with float16
    bnb_4bit_use_double_quant=True,
)

# 2. Get the model
model = AutoModelForCausalLM.from_pretrained(
    "./seu_diretorio_do_model", 
    quantization_config=bnb_config,
    device_map="auto" 
)

# See the difference
tamanho_bytes = model.get_memory_footprint()
print(f"Novo tamanho: {tamanho_bytes / 1024**2:.2f} MB")